In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [10]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Create a python function to extract the aws account Id from an ARN",
    },
    {
        "task": "Write a Json policy document that allows read only acces to specific S3 bucket",
    }
    
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [11]:
dataset = generate_dataset()
print(dataset)

[{'task': 'Write a regular expression to validate an AWS S3 bucket name according to AWS naming rules'}, {'task': 'Create a Python function that parses an AWS CloudFormation template JSON and returns all resource logical IDs'}, {'task': 'Write a JSON IAM policy document that allows EC2 instances to assume a role and read objects from a specific S3 bucket'}]


In [12]:
## guardar el conjunto de datos

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [13]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [14]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [15]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [16]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [17]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Validation Regex\n\nHere's a comprehensive regular expression to validate AWS S3 bucket names:\n\n```regex\n^[a-z0-9]([a-z0-9.-]{1,61}[a-z0-9])?$\n```\n\n## AWS S3 Bucket Naming Rules\n\nBefore the regex, let's review the official AWS requirements:\n\n1. **Length**: 3-63 characters\n2. **Characters**: Only lowercase letters (a-z), numbers (0-9), hyphens (-), and periods (.)\n3. **Start/End**: Must start and end with a lowercase letter or number\n4. **Consecutive periods**: Cannot contain consecutive periods (..)\n5. **Hyphen dash**: Cannot be formatted as an IP address\n6. **Format**: Cannot start with `xn--` (reserved for internationalized domain names)\n\n## Complete Validation Regex\n\nHere's a more comprehensive version that handles additional rules:\n\n```regex\n^(?!.*\\.\\.)[a-z0-9]([a-z0-9.-]{1,61}[a-z0-9])?$\n(?!xn--)\n```\n\n## Implementation Examples\n\n### JavaScript\n```javascript\nfunction validateS3BucketName(bucketName) {\n  cons

In [22]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


In [23]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

In [24]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [25]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)


Average score: 6.5


In [26]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Validation Regex\n\nHere's a comprehensive solution with explanations:\n\n## Basic Regex Pattern\n\n```regex\n^[a-z0-9]([a-z0-9\\-]{0,61}[a-z0-9])?$\n```\n\n## Complete Solution with Examples\n\n```python\nimport re\n\ndef validate_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates an AWS S3 bucket name according to AWS naming rules:\n    - 3-63 characters long\n    - Starts and ends with lowercase letter or number\n    - Contains only lowercase letters, numbers, and hyphens\n    - Cannot contain consecutive hyphens\n    - Cannot be formatted as an IP address\n    \"\"\"\n    \n    # Main pattern\n    pattern = r'^[a-z0-9]([a-z0-9\\-]{0,61}[a-z0-9])?$'\n    \n    # Check length (3-63 characters)\n    if len(bucket_name) < 3 or len(bucket_name) > 63:\n        return False\n    \n    # Check pattern\n    if not re.match(pattern, bucket_name):\n        return False\n    \n    # Cannot contain consecutive hyphens\n    if '--' in b